# Safari Compass Calibration

The Safari-Zone analog of the **Metronome Compass Calibration** notebook.  It
identifies the loaded seed, plans the manual advances needed to encounter a
Metang, identifies the *battle* seed from the safari encounter (bait / mud /
ball), checks how confident that identification is, saves the run, and feeds the
shared timer→frame calibration model.

## Two seeds, two kinds of "frame" (read this first)

Both seeds are fixed by **game-frame (clock) timing** — the timer precision we
calibrate:

- **Seed A** — the overworld stream: encounters, roamer relocation, Elm calls.
- **Seed B** — the battle stream: hits, crits, capture / flee odds.

An **advance frame** ("advance") is how many times a seed's state has been
advanced via `advance_rng`, driven by **player actions, not the clock**.  Section A
walks *Seed A's* advance frame (Elm calls + chatot flips + Sweet Scent) purely so
that we encounter a Metang — this does **not** affect Seed B or the calibration.
Calibration is the same timer(M)→Seed-B-frame fit as metronome; safari just
identifies Seed B differently and may carry a slightly different load-screen
offset, applied as a separate **safari offset** (β/slope stays from metronome).

## Sections
- **A** — identify Seed A (roamer + Elm), then plan the advances to a Metang.
- **B** — identify Seed B via safari compass, then a confidence / neighbor check.
- **C** — save the run to `data/safari_runs.jsonl`.
- **D** — analysis over the saved runs.
- **E** — apply the safari offset to `data/calibration_model.json` (offset only).

In [1]:
%load_ext autoreload
%autoreload 2
import datetime as dt

from utils.calibration_tools import (
    # Section A -- roamer routes + Elm seed identification (shared with metronome)
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    seed_for,                     # exact seed for a (datetime, delay) -- the intended target seed
    # Section C -- persist a safari run
    save_safari_run,
    # Section D -- analysis
    load_safari_runs,
    fit_safari_offset,
    # Section E -- apply the safari offset (deliberate; review-then-confirm)
    update_safari_offset,
)
from utils.safari_advance import (
    advance_context, context_from_row,
    identify_frame, prompt_target_frame, choose_target_frame,
    plan_advances, margin_guide, describe_plan,
)
from utils.safari_confidence import path_confidence, print_confidence

from claytonlib.compass import compass_safari, CompassSafariInput
from claytonlib.calibration import CalibrationModel
from claytonlib.safari import safari_pokemon_by_name
from claytonlib.chart import STRATEGY_ONLY_BALLS, CRITERIA_CAPTURE

## Section A — Identify Seed A, then plan to a Metang  (→ `a_seed`, advance plan)

Configure the target datetime/delay, the search window, and each roamer's **current**
route (before the reset).  After loading the save, this one cell:

1. **Identifies the seed** — roamer map + Elm phone (`identify_seed`): routes → Elm
   calls → (M) manual pick.
2. **Locates the current advance frame** from the Elm calls heard so far (1 Elm call =
   1 advance); the calls entered in step 1 are reused, prompting for more only if the
   frame is still ambiguous.
3. **Picks the target frame** (`choose_target_frame`):
   - Loaded the **target seed exactly** → `a_target_advances` (81 = shiny Metang),
     regardless of the toggle.
   - Else `a_use_inhouse=True` computes the nearest Metang frame from the Safari block
     config, **skipping frames whose Elm-call margin is an ambiguous run** (e.g. `[KKK]`);
     `a_use_inhouse=False` falls back to the Pokefinder handoff.
   - `a_blocks` are block **scores** (day-multiplier applied): plains/forest/peak/water.
4. **Plans the advances** — chatot flips (2 each) for the bulk, then a verifiable Elm-call
   margin, then Sweet Scent (`]!` in the guide).  A lone half/single flip is folded into a
   4–5 call margin instead.

In [6]:
# --- Section A config ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)   # <-- your load datetime
a_target_delay   = 681                                     # <-- your load delay
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset).  A roamer roams iff it appears here.
a_prev_routes    = {"r": 38, "e": 36, "l": 19}

# Advance-planning config
a_use_inhouse     = True          # True = compute the Metang frame in-house; False = Pokefinder handoff
a_target_advances = 81            # frame to hit IF we loaded the target seed EXACTLY (shiny Metang)
a_area            = "Mountain"    # Safari area you're hunting in
a_tod             = "morning"     # time of day: morning / day / night
a_blocks          = {"peak": 56}  # block SCORES (day-multiplier applied): plains/forest/peak/water

# 1. Identify Seed A: roamer routes -> Elm calls -> (M) manual pick.
a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, match_parity=a_match_parity,
)
a_seed = identify_seed(a_candidates, display_limit=a_display_limit)

# 2. Locate the current advance frame (reusing the Elm calls already entered above; 1 call = 1
#    advance).  Prompts for more only if the frame is still ambiguous.
a_rng_calls, a_elm = advance_context(a_seed["seed"], a_prev_routes, count=160)
a_current_frame = identify_frame(a_rng_calls, a_elm,
                                 observed=a_seed.get("elm_observed", ""), max_offset=15)

# 3. Pick the target frame: exact-seed hit -> a_target_advances (any toggle); else the in-house
#    Metang frame (a_use_inhouse=True; skips ambiguous-margin frames) or the Pokefinder prompt.
a_key_seed = seed_for(a_target_time, a_target_delay)
a_target_frame = choose_target_frame(
    a_seed["seed"], key_seed=a_key_seed, target_advances=a_target_advances,
    use_inhouse=a_use_inhouse, area=a_area, tod=a_tod, blocks=a_blocks,
    current_frame=a_current_frame, target="metang", rng_calls=a_rng_calls, elm=a_elm)

# 4. Plan the advances to the target frame.
a_plan  = plan_advances(a_current_frame, a_target_frame)
a_guide = margin_guide(a_rng_calls, a_elm, a_plan)
print(describe_plan(a_plan, a_guide))

KeyboardInterrupt: Interrupted by user

## Section B — Safari-compass Seed-B identification  (→ `b_matched`)

Drives the **calibrated** `compass_safari` (frame center from the model, ±kσ over
the RTC-second offsets), exactly as `expedition.compass_safari` does.  Walk the
safari encounter turn by turn — enter `m`/`b`/ball-shakes/`F`/`C` as you see them
— until the candidate set narrows.  Then a confidence check scans for other
nearby seeds that reproduce the same path (aliases), ranked by distance.

The boot key seed and initial time come straight from Section A's identified
`a_seed` (the loaded seed and its datetime) -- no need to re-enter them.  `b_M`
is the commanded countdown = `target_timer_delay + target_timer_calibration`.

In [3]:
# --- Section B: calibrated safari-compass target, then confidence / neighbor check ---
b_key_seed              = a_seed["seed"]                   # the loaded Seed A (from Section A)
b_initial_time          = a_seed["time"]                   # its datetime (from identify_seed)
b_target_timer_delay    = 249817                           # <-- commanded timer delay (ms)
b_target_timer_calibration = 0                             # <-- timer calibration (ms, signed)
b_max_target_seconds    = 600                              # <-- chart's max target (s)
b_pokemon_name          = "metang"
b_second_offsets        = (-1, 0, 1)   # cover off-by-one timer-start timing (the "3 seconds")
b_confidence_frame_range = 1000          # +/- frames to scan for path-aliases

b_M = b_target_timer_delay + b_target_timer_calibration
model = CalibrationModel.load_default()   # raw metronome fit (data/calibration_model.json)

b_inputs = CompassSafariInput.from_expedition_target(
    # Fold the fitted safari load-path offset into the frame center, exactly as
    # expedition.compass_safari does now -- otherwise Section B centers on the raw metronome
    # frame and drifts ~safari_offset frames off the expedition's target landing.  `model`
    # itself stays RAW so Section D's fit_safari_offset still measures against the metronome
    # fit (folding it there would collapse the offset to ~0 -- a double-correction).
    model=model.with_safari_offset(), M=b_M, initial_time=b_initial_time, key_seed=b_key_seed,
    max_target_seconds=b_max_target_seconds,
    pokemon=safari_pokemon_by_name(b_pokemon_name),
    strategy=STRATEGY_ONLY_BALLS, criteria=CRITERIA_CAPTURE,
    second_offsets=b_second_offsets, mass_cap=0.999,
)

# Interactive: enter the safari path as you play it out.
b_matched = compass_safari(b_inputs)

# Confidence / neighbor check: once a single seed remains, scan for aliases (other nearby seeds
# that reproduce the same path), ranked by distance.  The observed path comes straight from
# compass_safari -- no re-entry.
b_observed_path = b_matched.path
print(f"\nObserved path: {b_observed_path}")
if len(b_matched) == 1:
    b_seed = int(b_matched[0], 16)
    b_neighbors = path_confidence(b_inputs, b_seed, b_observed_path,
                                  frame_range=b_confidence_frame_range)
    print_confidence(b_neighbors)
else:
    b_seed = None
    print(f"{len(b_matched)} seeds still matched -- narrow further before trusting a single seed.")

=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit                  Metang is angry!
  M / a  Mud, crit (Anger)             Metang is beside itself with anger!
  b      Bait, no crit                 Metang is eating!
  B / e  Bait, crit (Eating)           Metang is busy eating!
  0      Ball, 0 shakes                Oh, no! The Pokémon broke free!
  1      Ball, 1 shake                 Aww! It appeared to be caught!
  2      Ball, 2 shakes                Aargh! Almost had it!
  3      Ball, 3 shakes                Shoot! It was so close, too!
  C      Captured (ends)               Gotcha! Metang was caught!
  F      Fled (ends)                   Metang fled!
  u      Undo last action              —
  ?x     Uncertain result              —
  J      Switch to Jane                —
  w      Widen window & re-apply path  —
  Spaces and commas in input are ignored.


Seeds: 1093 / 1093 remaining
Path:  (none)
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1.


>>  Bbbbbb



Seeds: 34 / 1093 remaining
Path:  Bbbbbb
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xE40E3A57    14935     -20      0    14.18%
   2. 0xE40E3A4B    14923     -32      0    12.95%
   3. 0xE40E3A95    14997     +42      0    11.62%
   4. 0xE40E3AA3    15011     +56      0     9.51%
   5. 0xE40E3A32    14898     -57      0     9.35%
  Most likely: 0xE40E3A57  P=14.18%  (timer on time)



>>  0



Seeds: 22 / 1093 remaining
Path:  Bbbbbb0
Balls: 29
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xE40E3A57    14935     -20      0    19.28%
   2. 0xE40E3A4B    14923     -32      0    17.59%
   3. 0xE40E3A95    14997     +42      0    15.79%
   4. 0xE40E3AA3    15011     +56      0    12.92%
   5. 0xE30E3A61    14945     -10     -1     5.02%
  Most likely: 0xE40E3A57  P=19.28%  (timer on time)


...


Seeds: 1 / 1093 remaining
Path:  Bbbbbb000000000000000MMmmmM
Balls: 15
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xE40E3A57    14935     -20      0   100.00%
  Most likely: 0xE40E3A57  P=100.00%  (timer on time)



>>  mmb



Seeds: 1 / 1093 remaining
Path:  Bbbbbb000000000000000MMmmmMmmb
Balls: 15
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xE40E3A57    14935     -20      0   100.00%
  Most likely: 0xE40E3A57  P=100.00%  (timer on time)



>>  mmC



Pokémon captured. 1 seed(s) matched this path:
Observed path: Bbbbbb000000000000000MMmmmMmmbmmC
  1. seed=0xE40E3A57  frame=14935  Δ=-20  δ=0s  P=100.00%

Observed path: Bbbbbb000000000000000MMmmmMmmbmmC
No other seed in the scanned window reproduces this path -- high confidence.


## Section C — Save the run  (→ `data/safari_runs.jsonl`)

Appends this run — the identified Seed B (only when a single seed matched), the
Section-A `a_seed` (so the offset fit has `F_a`), the observed path, the commanded
timer (`b_target_timer_delay`, passed straight in), and the calibrated landing
(frame / RTC second / δ) — via the existing `save_safari_run`.  Prompts only for a
**tag** and **notes**, then confirms before writing (every safari run is a fresh
boot, so those fields are fixed).  Saving does **not** touch the calibration model
(that's Section E).

In [4]:
# --- Section C: append this run to data/safari_runs.jsonl ---
# Uses b_target_timer_delay directly (no timer prompt); only prompts for tag, notes, and save.
# elm_calls / chatot_flips come from the Section A plan (recorded to study any correlation with
# the frame_delta / landing miss).
run_record = save_safari_run(b_matched, inputs=b_inputs, a_seed=a_seed, path=b_observed_path,
                             target_timer_delay=b_target_timer_delay,
                             elm_calls=a_plan.elm_before_scent, chatot_flips=a_plan.chatot_flips)

Inferred timer offset: on time (δ=0)  (frame 14935, RTC second 255).


Run tag [SCT3]:  
Notes:  



{
  "saved_at": "2026-09-14T11:31:10",
  "tag": "SCT3",
  "target_timer_delay": 249817,
  "path": "Bbbbbb000000000000000MMmmmMmmbmmC",
  "n_matched": 1,
  "matched_seeds": [
    "0xE40E3A57"
  ],
  "seed": 3826137687,
  "seed_hex": "0xE40E3A57",
  "delay": 14935,
  "frame": 14935,
  "target_frame": 14955,
  "frame_delta": -20,
  "second": 255,
  "second_offset": 0,
  "elm_calls": 3,
  "chatot_flips": 7.0,
  "a_seed": {
    "seed": 202244808,
    "seed_hex": "0x0C0E02C8",
    "time": "2025-07-24T14:45:55",
    "delay": 687,
    "sec_delta": 0,
    "delay_delta": 6,
    "r_route": 29,
    "e_route": 38,
    "l_route": 19,
    "rng_calls": 3,
    "elm": "KKPKKPKPPEKPKEE"
  },
  "notes": ""
}



Save this run? (y/n):  y


Saved to data/safari_runs.jsonl


## Section D — Analysis over `safari_runs.jsonl`

Sparse for now.  Shows the run count and previews the safari **offset** the
current runs imply against the deployed model (does *not* write it).  The
safari-vs-metronome offset measurement proper is tracked in `clayton-abf.10`.

In [7]:
# --- Section D: quick look at the collected safari runs ---
runs = load_safari_runs()
confident = [r for r in runs if r.get("seed") is not None]
print(f"{len(runs)} safari run(s) saved; {len(confident)} with a confident single seed.")

fit = fit_safari_offset(model)   # holds the model slope; median residual = the offset
if fit:
    print(f"Safari offset preview: {fit['offset']:+.2f} frames "
          f"(n={fit['n']}, std={fit['std']:.2f})  -- not written until Section E.")
else:
    print("No usable runs yet (need a_seed + a confident single seed).")

18 safari run(s) saved; 14 with a confident single seed.
Safari offset preview: -407.64 frames (n=14, std=190.37)  -- not written until Section E.


## Section E — Apply the safari offset  (→ `data/calibration_model.json`)

Re-fits the safari **offset only** (holding the metronome slope/β) from
`safari_runs.jsonl`, shows the old → new offset per model, and writes it **only
after you confirm**.  It sets a *separate* `safari_offset` field — the metronome
`alpha`/`beta` are untouched.  The expedition folds this offset into the frame
center for **all** safari scoring (`use_safari_offset`, default True), and so does
Section B above, so after changing it **re-run `precompute_chart()`** (a fast
incremental extend to the shifted frames) **and then `chart_report()`**.

In [8]:
# --- Section E: review the safari offset re-fit, then write it only if confirmed ---
new_models = update_safari_offset()


[linear] safari_offset -435.27 -> -407.64 frames  (n=14, std=190.37)
[quad] safari_offset -460.97 -> -414.04 frames  (n=14, std=202.13)

*** This shifts the safari load-path frame center. The expedition applies it to ALL safari scoring (use_safari_offset, default True), so RE-RUN precompute_chart() to extend the canon to the shifted frames, then chart_report(). (Set expedition.use_safari_offset=False to score against the raw metronome fit instead.) ***


Write the safari offset? (y/n):  n


Safari offset not updated.
